# ETL — Olist (CSV) → SQLite

**Escopo deste notebook:** apenas **consumo das fontes**, **tratamento de tipos** (conforme `dicionario.xlsx`) e **carga no banco** usado pelo **Power BI** para indicadores e relacionamentos.

**Não** repetimos strings de conexão aqui: a conexão com o SQLite fica centralizada em `utils.db` (`with_connection` / `DB_PATH`).

**Fontes:** CSVs na pasta `archive` · **Tipos e descrições:** `dicionario.xlsx` · **Destino:** `database.db` na raiz do projeto.

## Governança e rastreabilidade

| Item | Detalhe |
|------|---------|
| Origem | Arquivos `olist_*.csv` e `product_category_name_translation.csv` |
| Dicionário | `dicionario.xlsx` (abas por entidade: colunas `Campo`, `Tipo`, `Descrição`) |
| Destino | `database.db` — tabelas homônimas aos CSVs (sem extensão) |
| Integridade | Chaves estrangeiras no SQLite alinhadas ao modelo Olist (pedidos → clientes; itens → pedidos, produtos, vendedores; pagamentos e avaliações → pedidos) |
| Regra extra | Avaliações: remoção de `review_id` duplicado (814 linhas no arquivo público), mantendo a primeira ocorrência |
| Reprodutibilidade | Pipeline em `utils/etl_olist.py`; parâmetros `archive` e `dicionario` podem ser alterados abaixo |

## Fase 1 — Configuração do ambiente

Ajuste os caminhos se o projeto ou os arquivos estiverem em outro diretório.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "utils").is_dir():
    ROOT = Path(r"c:\Users\lopes\OneDrive\Área de Trabalho\fiap")
    sys.path.insert(0, str(ROOT))

from utils import (
    DB_PATH,
    DEFAULT_ARCHIVE,
    DEFAULT_DICIONARIO,
    load_dicionario,
    resumo_dicionario_markdown,
    run_olist_etl,
    with_connection,
)

# Pastas dos dados (edite se necessário)
ARCHIVE = Path(r"C:\Users\lopes\Downloads\archive")
DICIONARIO_XLSX = Path(r"C:\Users\lopes\Downloads\dicionario.xlsx")

print("Projeto:", ROOT)
print("SQLite:", DB_PATH)
print("Pasta CSV:", ARCHIVE, "— existe:", ARCHIVE.is_dir())
print("Dicionário:", DICIONARIO_XLSX, "— existe:", DICIONARIO_XLSX.is_file())

Projeto: c:\Users\lopes\OneDrive\Área de Trabalho\fiap
SQLite: C:\Users\lopes\OneDrive\Área de Trabalho\fiap\database.db
Pasta CSV: C:\Users\lopes\Downloads\archive — existe: True
Dicionário: C:\Users\lopes\Downloads\dicionario.xlsx — existe: True


## Fase 2 — Inspeção do dicionário de tipos

Cada aba do Excel define os **campos**, **tipos** (`string`, `int`, `float`, `datetime`) e **descrição** usados na transformação.

In [2]:
from IPython.display import display

dic = load_dicionario(DICIONARIO_XLSX)
print(resumo_dicionario_markdown(DICIONARIO_XLSX))
print()
# Exemplo: aba "Pedidos" (fallback: segunda aba na ordem usual do arquivo)
mostrado = False
for nome, df_aba in dic.items():
    if nome.strip().lower() == "pedidos":
        display(df_aba)
        mostrado = True
        break
if not mostrado and len(dic) >= 2:
    display(list(dic.values())[1])

| Aba | Campos |
|-----|--------|
| Clientes | 5 |
| Pedidos | 8 |
| Itens do Pedido | 7 |
| Pagamentos | 5 |
| Avaliações | 7 |
| Produtos | 9 |
| Vendedores | 4 |
| Geolocalização | 5 |
| Tradução de Categorias | 2 |



,Campo,Tipo,Descrição
0,order_id,string,ID do pedido
1,customer_id,string,FK para cliente
2,order_status,string,"Status (delivered, shipped, canceled…)"
3,order_purchase_timestamp,datetime,Data da compra
4,order_approved_at,datetime,Data de aprovação
5,order_delivered_carrier_date,datetime,Data envio transportadora
6,order_delivered_customer_date,datetime,Data entrega ao cliente
7,order_estimated_delivery_date,datetime,Data estimada


## Fase 3 — ETL completo (Extract → Transform → Load)

A lógica padronizada está em **`utils/etl_olist.py`**:

1. **Extract:** leitura de cada CSV na ordem correta de dependência.
2. **Transform:** renomeação de colunas de produtos (`lenght` → `length`), aplicação dos tipos do dicionário, datas em texto ISO para o SQLite.
3. **Load:** `DROP` das tabelas (com `PRAGMA foreign_keys=OFF`), `CREATE` com FKs, `INSERT` via `pandas.to_sql`.

Para testar só leitura e tipos sem gravar no banco, use `dry_run=True` em `run_olist_etl`.

In [3]:
# Carga completa (substitui o conteúdo das tabelas Olist no database.db)
contagens = run_olist_etl(archive=ARCHIVE, dicionario=DICIONARIO_XLSX)
pd.Series(contagens, name="linhas_inseridas")

Carga: olist_geolocation_dataset — 1000163 linhas
Carga: product_category_name_translation — 71 linhas
Carga: olist_customers_dataset — 99441 linhas
Carga: olist_sellers_dataset — 3095 linhas
Carga: olist_products_dataset — 32951 linhas
Carga: olist_orders_dataset — 99441 linhas
Carga: olist_order_items_dataset — 112650 linhas
Carga: olist_order_payments_dataset — 103886 linhas
Carga: olist_order_reviews_dataset — 98410 linhas


olist_geolocation_dataset            1000163
product_category_name_translation         71
olist_customers_dataset                99441
olist_sellers_dataset                   3095
olist_products_dataset                 32951
olist_orders_dataset                   99441
olist_order_items_dataset             112650
olist_order_payments_dataset          103886
olist_order_reviews_dataset            98410
Name: linhas_inseridas, dtype: int64

## Fase 4 — Conferência pós-carga

Consulta simples usando a mesma conexão utilitária (sem SQL explícito de conexão no notebook).

In [4]:
with with_connection() as conn:
    cur = conn.execute(
        """SELECT name FROM sqlite_master WHERE type='table'
        AND (name LIKE 'olist%' OR name LIKE 'product%') ORDER BY name"""
    )
    tabelas = [r[0] for r in cur.fetchall()]
    contagens = {}
    for t in tabelas:
        n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        contagens[t] = n

pd.Series(contagens, name="linhas")

olist_customers_dataset                99441
olist_geolocation_dataset            1000163
olist_order_items_dataset             112650
olist_order_payments_dataset          103886
olist_order_reviews_dataset            98410
olist_orders_dataset                   99441
olist_products_dataset                 32951
olist_sellers_dataset                   3095
product_category_name_translation         71
Name: linhas, dtype: int64

## Power BI e entregáveis

- **Conectar:** *Obter dados* → *SQLite* → informar o caminho de `database.db` (use o caminho absoluto exibido na célula de configuração).
- **Relacionamentos:** alinhar às chaves do diagrama Olist (`order_id`, `customer_id`, `product_id`, `seller_id`, `zip_code_prefix` entre geolocalização e clientes/vendedores).

**Links do grupo:**

- Repositório GitHub: [TECH_CHALLENGE_GP18](https://github.com/MarcioBKN/TECH_CHALLENGE_GP18)
- Apresentação: `https://...`
- Vídeo executivo (até 5 min): `https://...`